# Example 12.2 / 2: Connect the Route with a Load Flow or MTZ Order

Companion notebook for **12.2 Routing Models** in the AMPL lab Documentation, section "Connect the Route with a Load Flow or MTZ Order". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Connect the Route with a Load Flow or MTZ Order

Two families of rows connect every customer to the depot. A **single-commodity flow** sends the
customers' demand out of the depot along the selected arcs. With demand $d_j$ and vehicle
capacity $Q$, `Load[i,j]` is the quantity still on the vehicle when it travels from `i` to `j`;
it may be positive only on a selected arc, $Load_{ij} \le Q \, Use_{ij}$, and every customer
keeps its demand, so inflow minus outflow equals $d_j$. The depot balance follows from the
others. **MTZ order variables** give each customer a position after the depot and require the
position to grow by at least 1 along every selected arc that does not end at the depot, which no
cycle avoiding the depot can satisfy.

The model holds both families; `drop()` and `restore()` on a constraint switch it off and on.
It extends the degree model above, so data.py adds `demand` and `capacity` to the same five
nodes. The first run drops the MTZ rows and tries capacities 15 and 12, where 12 is below the
total demand 14. It then follows the successors from the depot to rebuild the route and checks
that every customer appears once:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set NODES ordered;                            # depot first, then customers
set ARCS within {NODES, NODES};               # allowed directed arcs
param cost {ARCS} >= 0;                       # travel cost per arc
param demand {NODES} >= 0;                    # load delivered to each customer
param capacity > 0;                           # vehicle capacity Q

var Use {ARCS} binary;                        # 1 if the tour uses arc (i,j)
var Load {ARCS} >= 0;                         # load carried on each arc
var Order {NODES} >= 0, <= card(NODES) - 1;   # MTZ position after the depot

minimize TravelCost: sum {(i,j) in ARCS} cost[i,j] * Use[i,j];  # total travel cost

subject to Leave {i in NODES}: sum {(i,j) in ARCS} Use[i,j] = 1;  # one departure from every node
subject to Enter {j in NODES}: sum {(i,j) in ARCS} Use[i,j] = 1;  # one arrival at every node

# single-commodity flow rows: load only on selected arcs, up to Q
subject to LinkLoad {(i,j) in ARCS}: Load[i,j] <= capacity * Use[i,j];
subject to Deliver {j in NODES: j <> first(NODES)}:  # flow balance: each customer keeps its demand
    sum {(i,j) in ARCS} Load[i,j] - sum {(j,k) in ARCS} Load[j,k] = demand[j];

# MTZ order rows
subject to MTZ {(i,j) in ARCS: j <> first(NODES)}:
    Order[j] >= Order[i] + 1 - (card(NODES) - 1) * (1 - Use[i,j]);

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
nodes = ["depot", "c1", "c2", "c3", "c4"]
travel = {                      # same five nodes as above
    "depot": [0, 2, 2, 9, 9],
    "c1":    [2, 0, 1, 8, 9],
    "c2":    [2, 2, 0, 9, 8],
    "c3":    [9, 8, 9, 0, 1],
    "c4":    [9, 9, 7, 1, 0],
}
ampl.set["NODES"] = nodes
ampl.set["ARCS"] = [(i, j) for i in nodes for j in nodes if i != j]
ampl.param["cost"] = {(i, j): travel[i][k]
                      for i in nodes for k, j in enumerate(nodes) if i != j}
ampl.param["demand"] = {"depot": 0, "c1": 3, "c2": 4, "c3": 2, "c4": 5}
ampl.param["capacity"] = 15

In [5]:
# run.py
def route_from_depot():
    succ = {i: j for (i, j), used in ampl.var["Use"].to_dict().items() if used > 0.5}
    route = ["depot"]
    while len(route) == 1 or route[-1] != "depot":
        route.append(succ[route[-1]])
    return route

ampl.con["MTZ"].drop()
for capacity in [15, 12]:
    ampl.param["capacity"] = capacity
    ampl.solve(solver="highs", verbose=False)
    print("capacity", capacity, ampl.solve_result)
    if ampl.solve_result == "solved":
        load = {arc: q for arc, q in ampl.var["Load"].to_dict().items() if q > 1e-6}
        route = route_from_depot()
        print(" cost", ampl.obj["TravelCost"].value(), "route", route)
        print(" loads", load)
        print(" every customer once:", sorted(route[1:-1]) == sorted(nodes[1:]))

capacity 15 solved
 cost 20.0 route ['depot', 'c1', 'c3', 'c4', 'c2', 'depot']
 loads {('c1', 'c3'): 11, ('c3', 'c4'): 9, ('c4', 'c2'): 4, ('depot', 'c1'): 14}
 every customer once: True
capacity 12 infeasible


### Pure AMPL: model.run

Pure AMPL, with the same model; data.dat is the file from "Assign Arcs with Degree Equations"
plus the lines `param demand := depot 0  c1 3  c2 4  c3 2  c4 5;` and `param capacity := 15;`:

In [6]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [7]:
# data.py
nodes = ["depot", "c1", "c2", "c3", "c4"]
travel = {                      # same five nodes as above
    "depot": [0, 2, 2, 9, 9],
    "c1":    [2, 0, 1, 8, 9],
    "c2":    [2, 2, 0, 9, 8],
    "c3":    [9, 8, 9, 0, 1],
    "c4":    [9, 9, 7, 1, 0],
}
ampl.set["NODES"] = nodes
ampl.set["ARCS"] = [(i, j) for i in nodes for j in nodes if i != j]
ampl.param["cost"] = {(i, j): travel[i][k]
                      for i in nodes for k, j in enumerate(nodes) if i != j}
ampl.param["demand"] = {"depot": 0, "c1": 3, "c2": 4, "c3": 2, "c4": 5}
ampl.param["capacity"] = 15

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
drop MTZ;
for {cap in {15, 12}} {
    let capacity := cap;
    solve;
    printf "capacity %d %s\n", cap, solve_result;
    if solve_result = "solved" then {
        display TravelCost;
        display {(i,j) in ARCS: Load[i,j] > 1e-6} Load[i,j];
    }
}

capacity 15 solved
TravelCost = 20

Load[i,j] :=
c1    c3   11
c3    c4    9
c4    c2    4
depot c1   14
;


suffix dunbdd OUT;
capacity 12 infeasible


### Another way in amplpy

With capacity 15 the route is depot-c1-c3-c4-c2-depot at cost 20. The load falls 14, 11, 9, 4
along it, so the depot sends exactly the total demand 14 and no arc carries more than 15. With
capacity 12 no single route can carry the demand, and the solver reports the model infeasible
instead of returning a cycle cover.

MTZ rows are compact but often weak. Single- or multi-commodity flow formulations are larger and
usually stronger. Exponential families of subtour constraints can be generated lazily instead. To
compare two formulations, solve the LP relaxation of each: the AMPL option `relax_integrality`
set to 1 treats every integer variable as continuous, and the LP optimum is a lower bound on the
integer optimum (the **root bound**). This run, with the same model and data, keeps one family at
a time:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
nodes = ["depot", "c1", "c2", "c3", "c4"]
travel = {                      # same five nodes as above
    "depot": [0, 2, 2, 9, 9],
    "c1":    [2, 0, 1, 8, 9],
    "c2":    [2, 2, 0, 9, 8],
    "c3":    [9, 8, 9, 0, 1],
    "c4":    [9, 9, 7, 1, 0],
}
ampl.set["NODES"] = nodes
ampl.set["ARCS"] = [(i, j) for i in nodes for j in nodes if i != j]
ampl.param["cost"] = {(i, j): travel[i][k]
                      for i in nodes for k, j in enumerate(nodes) if i != j}
ampl.param["demand"] = {"depot": 0, "c1": 3, "c2": 4, "c3": 2, "c4": 5}
ampl.param["capacity"] = 15

In [11]:
# run.py
families = {"degree only": [], "MTZ": ["MTZ"], "flow": ["LinkLoad", "Deliver"]}
for label, keep in families.items():
    for name in ["MTZ", "LinkLoad", "Deliver"]:
        if name in keep:
            ampl.con[name].restore()
        else:
            ampl.con[name].drop()
    ampl.option["relax_integrality"] = 1
    ampl.solve(solver="highs", verbose=False)
    bound = ampl.obj["TravelCost"].value()
    ampl.option["relax_integrality"] = 0
    ampl.solve(solver="highs", verbose=False)
    print(f"{label}: LP bound {bound:.4g}, integer {ampl.obj['TravelCost'].value():g}")

degree only: LP bound 7, integer 7
MTZ: LP bound 13.5, integer 20
flow: LP bound 13.07, integer 20


### Pure AMPL: model.run

Pure AMPL:

In [12]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [13]:
# data.py
nodes = ["depot", "c1", "c2", "c3", "c4"]
travel = {                      # same five nodes as above
    "depot": [0, 2, 2, 9, 9],
    "c1":    [2, 0, 1, 8, 9],
    "c2":    [2, 2, 0, 9, 8],
    "c3":    [9, 8, 9, 0, 1],
    "c4":    [9, 9, 7, 1, 0],
}
ampl.set["NODES"] = nodes
ampl.set["ARCS"] = [(i, j) for i in nodes for j in nodes if i != j]
ampl.param["cost"] = {(i, j): travel[i][k]
                      for i in nodes for k, j in enumerate(nodes) if i != j}
ampl.param["demand"] = {"depot": 0, "c1": 3, "c2": 4, "c3": 2, "c4": 5}
ampl.param["capacity"] = 15

In [14]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
param bound;  # LP bound of the current formulation
drop MTZ; drop LinkLoad; drop Deliver;
option relax_integrality 1; solve; let bound := TravelCost;
option relax_integrality 0; solve;
printf "degree only: LP bound %.4g, integer %g\n", bound, TravelCost;
restore MTZ;
option relax_integrality 1; solve; let bound := TravelCost;
option relax_integrality 0; solve;
printf "MTZ: LP bound %.4g, integer %g\n", bound, TravelCost;
drop MTZ; restore LinkLoad; restore Deliver;
option relax_integrality 1; solve; let bound := TravelCost;
option relax_integrality 0; solve;
printf "flow: LP bound %.4g, integer %g\n", bound, TravelCost;

degree only: LP bound 7, integer 7
MTZ: LP bound 13.5, integer 20
flow: LP bound 13.07, integer 20
